# GraphRAG & Knowledge Graphs

> **Track:** AI Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
Naive RAG retrieves isolated text chunks. **GraphRAG** augments retrieval with a **knowledge graph** — a structured representation of entities and their relationships — enabling multi-hop reasoning, relationship-aware answers, and community-level summarization.

### What You'll Learn
- Limitations of vector-only RAG
- Knowledge graph construction from text
- Entity and relation extraction
- Graph-based retrieval with NetworkX
- Hybrid graph + vector search
- Microsoft GraphRAG community summarization approach

```bash
pip install networkx openai chromadb matplotlib
```

In [ ]:
# Setup: imports and clients
import json
import os
from collections import defaultdict
from typing import Any

import networkx as nx
import matplotlib.pyplot as plt
import openai

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", "YOUR_KEY"))

# Sample corpus: a small set of documents about a fictional tech company
CORPUS = [
    "Alice is the CEO of TechCorp. She founded the company in 2015.",
    "TechCorp acquired DataStream in 2022 for $500M. DataStream was founded by Bob.",
    "Bob now leads the AI division at TechCorp after the acquisition.",
    "Alice and Bob co-authored a paper on distributed ML systems.",
    "TechCorp's main product is CloudML, a platform for MLOps.",
    "CloudML integrates with Kubernetes and Apache Spark for large-scale training.",
]

print(f"Corpus size: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i}] {doc}")

## 1. Entity & Relation Extraction

The first step in building a knowledge graph is extracting **entities** (people, organizations, products) and **relations** (founded, acquired, leads) from raw text using an LLM.

In [ ]:
# Extract entities and relations from each document using LLM

EXTRACTION_PROMPT = """
Extract entities and relationships from the text below.
Return JSON with this structure:
{"entities": [{"name": str, "type": str}], "relations": [{"source": str, "relation": str, "target": str}]}
Types: PERSON, ORGANIZATION, PRODUCT, TECHNOLOGY
Text: {text}
"""

def extract_graph_elements(text: str) -> dict[str, list]:
    """Use LLM to extract entities and relations from a text chunk."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# Mock extraction results to avoid API calls in demo
MOCK_EXTRACTIONS = [
    {"entities": [{"name": "Alice", "type": "PERSON"}, {"name": "TechCorp", "type": "ORGANIZATION"}],
     "relations": [{"source": "Alice", "relation": "CEO_OF", "target": "TechCorp"}, {"source": "Alice", "relation": "FOUNDED", "target": "TechCorp"}]},
    {"entities": [{"name": "TechCorp", "type": "ORGANIZATION"}, {"name": "DataStream", "type": "ORGANIZATION"}, {"name": "Bob", "type": "PERSON"}],
     "relations": [{"source": "TechCorp", "relation": "ACQUIRED", "target": "DataStream"}, {"source": "Bob", "relation": "FOUNDED", "target": "DataStream"}]},
    {"entities": [{"name": "Bob", "type": "PERSON"}, {"name": "TechCorp", "type": "ORGANIZATION"}],
     "relations": [{"source": "Bob", "relation": "LEADS", "target": "TechCorp"}]},
    {"entities": [{"name": "Alice", "type": "PERSON"}, {"name": "Bob", "type": "PERSON"}],
     "relations": [{"source": "Alice", "relation": "CO_AUTHORED_WITH", "target": "Bob"}]},
    {"entities": [{"name": "TechCorp", "type": "ORGANIZATION"}, {"name": "CloudML", "type": "PRODUCT"}],
     "relations": [{"source": "TechCorp", "relation": "MAKES", "target": "CloudML"}]},
    {"entities": [{"name": "CloudML", "type": "PRODUCT"}, {"name": "Kubernetes", "type": "TECHNOLOGY"}, {"name": "Apache Spark", "type": "TECHNOLOGY"}],
     "relations": [{"source": "CloudML", "relation": "INTEGRATES_WITH", "target": "Kubernetes"}, {"source": "CloudML", "relation": "INTEGRATES_WITH", "target": "Apache Spark"}]},
]

print("Extraction complete (using mock data).")
print(f"Total relations extracted: {sum(len(e['relations']) for e in MOCK_EXTRACTIONS)}")

## 2. Build the Knowledge Graph

We load all extracted entities and relations into a **NetworkX directed graph**. Each node is an entity; each edge is a labeled relation.

In [ ]:
# Build a NetworkX knowledge graph from extracted elements

def build_knowledge_graph(extractions: list[dict]) -> nx.DiGraph:
    """Construct a directed knowledge graph from extraction results."""
    G = nx.DiGraph()
    for extraction in extractions:
        for entity in extraction["entities"]:
            G.add_node(entity["name"], type=entity["type"])
        for rel in extraction["relations"]:
            G.add_edge(rel["source"], rel["target"], relation=rel["relation"])
    return G

G = build_knowledge_graph(MOCK_EXTRACTIONS)

# Visualize the graph
fig, ax = plt.subplots(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42)
node_colors = {"PERSON": "#4CAF50", "ORGANIZATION": "#2196F3", "PRODUCT": "#FF9800", "TECHNOLOGY": "#9C27B0"}
colors = [node_colors.get(G.nodes[n].get("type", "PERSON"), "gray") for n in G.nodes]
nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors, node_size=2000,
        font_size=9, arrows=True, edge_color="gray", arrowsize=20)
edge_labels = nx.get_edge_attributes(G, 'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)
ax.set_title("TechCorp Knowledge Graph")
plt.tight_layout()
plt.show()

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 3. Graph-Based Retrieval

For a query like "Who works at TechCorp?", we can:
1. Extract the entity from the query ("TechCorp")
2. Traverse the graph to find related entities
3. Return context that captures multi-hop connections

In [ ]:
# Graph traversal for multi-hop retrieval

def get_entity_subgraph_context(G: nx.DiGraph, entity: str, hops: int = 2) -> str:
    """Get a textual description of an entity's neighborhood in the graph."""
    if entity not in G:
        return f"Entity '{entity}' not found in knowledge graph."

    # Collect all nodes within N hops
    nodes_in_scope = {entity}
    frontier = {entity}
    for _ in range(hops):
        new_frontier = set()
        for node in frontier:
            new_frontier.update(G.predecessors(node))
            new_frontier.update(G.successors(node))
        frontier = new_frontier - nodes_in_scope
        nodes_in_scope.update(frontier)

    # Build a text summary of relations involving these nodes
    context_lines: list[str] = []
    subgraph = G.subgraph(nodes_in_scope)
    for src, tgt, data in subgraph.edges(data=True):
        context_lines.append(f"{src} --[{data['relation']}]--> {tgt}")

    return f"Knowledge graph context (entity: {entity}, {hops}-hop):\n" + "\n".join(context_lines)

# Test multi-hop retrieval
print(get_entity_subgraph_context(G, "TechCorp", hops=2))
print()
print(get_entity_subgraph_context(G, "Alice", hops=2))

## 4. Hybrid Graph + Vector RAG

The most powerful approach combines:
- **Vector search** — for semantic similarity retrieval
- **Graph traversal** — for structured relationship context
- Both results are merged into the LLM prompt

In [ ]:
# Hybrid GraphRAG: combine vector retrieval + graph traversal

def hybrid_graphrag_query(
    query: str,
    G: nx.DiGraph,
    corpus: list[str],
    entity_hints: list[str]
) -> str:
    """Query using both text chunks and knowledge graph context."""
    # 1. Vector retrieval (simplified: use substring match as stub)
    query_lower = query.lower()
    vector_chunks = [doc for doc in corpus if any(word in doc.lower() for word in query_lower.split())]

    # 2. Graph retrieval for each hinted entity
    graph_contexts: list[str] = []
    for entity in entity_hints:
        if entity in G:
            graph_contexts.append(get_entity_subgraph_context(G, entity, hops=2))

    # 3. Build augmented prompt
    prompt_parts = ["=== Retrieved Text Chunks ==="] + vector_chunks
    if graph_contexts:
        prompt_parts += ["\n=== Knowledge Graph Context ==="] + graph_contexts
    prompt_parts += [f"\nQuestion: {query}", "Answer:"]
    full_prompt = "\n".join(prompt_parts)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": full_prompt}]
    )
    return response.choices[0].message.content

print("Hybrid GraphRAG function ready.")
print("Example usage:")
print("  hybrid_graphrag_query('Who leads AI at TechCorp?', G, CORPUS, ['TechCorp', 'Bob'])")
print("\nUncomment to run (requires OPENAI_API_KEY):")
print("# answer = hybrid_graphrag_query('Who leads AI at TechCorp and what did they build?', G, CORPUS, ['TechCorp', 'Bob'])")
print("# print(answer)")

## Exercises

1. **Extend the corpus**: Add 3 more documents about TechCorp products or people. Re-run the extraction and graph building steps. Observe how the graph grows and new relationships appear.

2. **Community detection**: Use `networkx.algorithms.community.greedy_modularity_communities(G.to_undirected())` to detect communities in the graph. Generate a summary for each community using an LLM and test whether community-level summaries help answer higher-level questions.

3. **Relationship path finding**: Implement a function `find_connection_path(G, entity_a, entity_b)` that uses `nx.shortest_path` to find the relationship chain between two entities, then generates a natural-language explanation of how they are connected.